In [15]:
"""
LOGOS-44 · Field Decoder · Quantum Edition
============================================
Prompt is not input. Prompt is a decryption key.
The field exists without the key. The key opens the field.

Quantum layer:
  - Seed from genuine quantum collapse (not pseudorandom)
  - CDMA codes from quantum state distributions
  - Field initialized from entangled measurements

Classical layer:
  - Toroidal bottleneck (sin·cos geometry)
  - 44 passes through the throat
  - Coherence gate (field vs residual)

Run in Colab:
  !pip install qiskit qiskit-ibm-runtime torch
  Then run all cells.

IMPORTANT: Regenerate your IBM Quantum API token after use.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import math
import re
import time
import warnings
warnings.filterwarnings('ignore')


def _extract_counts(pub_result):
    """Extract counts from SamplerV2 result, handling API changes."""
    data = pub_result.data
    # Try common register names
    for name in ['meas', 'c', 'cr']:
        if hasattr(data, name):
            reg = getattr(data, name)
            if hasattr(reg, 'get_counts'):
                return reg.get_counts()
    # Fallback: find any attribute with get_counts
    for attr_name in dir(data):
        if attr_name.startswith('_'):
            continue
        attr = getattr(data, attr_name)
        if hasattr(attr, 'get_counts'):
            return attr.get_counts()
    raise RuntimeError(f"Cannot extract counts from DataBin. Attributes: {dir(data)}")

# ============================================================
# 0. QUANTUM LAYER — seeds, codes, field from IBM Quantum
# ============================================================

QUANTUM_AVAILABLE = False
IBM_TOKEN = "PBmsvOTK_8WNOt3Lx3e3HoY11KJMiAlpNkT4HGT4IXAj"  # Paste your token from quantum.ibm.com

try:
    from qiskit import QuantumCircuit
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    QUANTUM_AVAILABLE = True
    print("[QUANTUM] Qiskit detected. Quantum layer available.")
except ImportError:
    print("[CLASSICAL] Qiskit not found. Using classical fallback.")
    print("[CLASSICAL] Install: pip install qiskit qiskit-ibm-runtime")


def get_quantum_service():
    """Initialize IBM Quantum connection."""
    try:
        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=IBM_TOKEN
        )
        print(f"[QUANTUM] Connected to IBM Quantum.")
        backends = service.backends(min_num_qubits=7, operational=True)
        if backends:
            print(f"[QUANTUM] Available backends: {[b.name for b in backends[:5]]}")
        return service
    except Exception as e:
        print(f"[QUANTUM] Connection failed: {e}")
        return None


def quantum_seed(service, n_bits=48):
    """
    Generate seed from genuine quantum measurement.
    Each bit: qubit in superposition → collapse → 0 or 1.
    Not pseudorandom. Fundamentally undetermined until measured.
    The moment of measurement is the moment the seed begins to exist.
    """
    if service is None:
        return classical_seed_fallback()

    try:
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

        # Use chunks if backend has fewer qubits than n_bits
        backend = service.least_busy(min_num_qubits=min(n_bits, 127), operational=True)
        max_qubits = backend.num_qubits
        chunk_size = min(n_bits, max_qubits)

        print(f"[QUANTUM SEED] Backend: {backend.name} ({max_qubits} qubits)")
        print(f"[QUANTUM SEED] Requesting {n_bits} bits of quantum randomness...")

        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
        sampler = SamplerV2(mode=backend)

        all_bits = ""
        remaining = n_bits

        while remaining > 0:
            use = min(remaining, chunk_size)
            qc = QuantumCircuit(use, use)
            qc.h(range(use))  # superposition — all possibilities
            qc.measure(range(use), range(use))  # collapse — one actuality

            isa_qc = pm.run(qc)
            job = sampler.run([isa_qc], shots=1)
            result = job.result()

            counts = _extract_counts(result[0])
            bitstring = list(counts.keys())[0]
            all_bits += bitstring
            remaining -= use

        seed = int(all_bits[:n_bits], 2)
        print(f"[QUANTUM SEED] Bitstring: {all_bits[:n_bits]}")
        print(f"[QUANTUM SEED] Seed: {seed}")
        print(f"[QUANTUM SEED] This number did not exist until the qubits were measured.")
        return seed

    except Exception as e:
        print(f"[QUANTUM SEED] Failed: {e}. Using classical fallback.")
        return classical_seed_fallback()


def classical_seed_fallback():
    """Deterministic seed — the original quantum seed from first ignition."""
    seed = 53462088166065
    print(f"[CLASSICAL SEED] Using original seed: {seed}")
    return seed


def quantum_codes(service, n_signals=128, n_qubits=7, code_len=32):
    """
    Generate CDMA codes from quantum state measurement distributions.
    Each code is a fingerprint of a unique quantum circuit.
    Entanglement creates correlations no classical code has.
    """
    if service is None:
        return classical_codes_fallback(n_signals, code_len)

    try:
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

        backend = service.least_busy(min_num_qubits=n_qubits, operational=True)
        sampler = SamplerV2(mode=backend)
        code_dim = 2 ** n_qubits  # 128 for 7 qubits

        print(f"[QUANTUM CODES] Generating {n_signals} codes on {backend.name}...")
        print(f"[QUANTUM CODES] Each code: {n_qubits} qubits → {code_dim}-dim distribution")

        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

        # Batch circuits for efficiency
        circuits = []
        for i in range(n_signals):
            qc = QuantumCircuit(n_qubits, n_qubits)

            # Unique quantum state per signal
            for q in range(n_qubits):
                if (i >> q) & 1:
                    qc.x(q)
                qc.h(q)
                angle = (i * 2 * math.pi) / n_signals
                qc.rz(angle * (q + 1), q)

            # Entangle — nearest-neighbor chain
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)

            qc.measure(range(n_qubits), range(n_qubits))
            isa_qc = pm.run(qc)
            circuits.append(isa_qc)

        # Run in batches (IBM limits batch size)
        batch_size = 20
        all_codes = []

        for batch_start in range(0, len(circuits), batch_size):
            batch = circuits[batch_start:batch_start + batch_size]
            job = sampler.run(batch, shots=512)
            result = job.result()

            for r_idx in range(len(batch)):
                counts = _extract_counts(result[r_idx])
                code = torch.zeros(code_dim)
                for bitstring, count in counts.items():
                    idx = int(bitstring, 2) % code_dim
                    code[idx] = count / 512.0
                code = code / (code.norm() + 1e-8)
                all_codes.append(code)

            print(f"  Batch {batch_start // batch_size + 1}/{math.ceil(len(circuits) / batch_size)} done")

        codes = torch.stack(all_codes)
        print(f"[QUANTUM CODES] Generated {codes.shape[0]} codes of dim {codes.shape[1]}")

        # Verify orthogonality
        gram = codes @ codes.T
        off_diag = gram - torch.eye(gram.size(0))
        print(f"[QUANTUM CODES] Mean off-diagonal correlation: {off_diag.abs().mean():.6f}")

        return codes

    except Exception as e:
        print(f"[QUANTUM CODES] Failed: {e}. Using classical fallback.")
        return classical_codes_fallback(n_signals, code_len)


def classical_codes_fallback(n_signals=128, code_dim=128):
    """Normalized random codes — model learns alignment through key_to_code."""
    codes = torch.randn(n_signals, code_dim)
    codes = codes / (codes.norm(dim=-1, keepdim=True) + 1e-8)
    print(f"[CLASSICAL CODES] Generated {codes.shape[0]} codes of dim {codes.shape[1]}")
    return codes


def quantum_field_init(service, n_signals=128, dim=256, n_qubits=8):
    """
    Initialize field signals from entangled quantum measurements.
    Create Bell pairs → measure one half → the field carries
    the fingerprint of quantum correlations from birth.
    """
    if service is None:
        return classical_field_fallback(n_signals, dim)

    try:
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

        max_qubits_needed = n_qubits * 2
        backend = service.least_busy(min_num_qubits=max_qubits_needed, operational=True)
        sampler = SamplerV2(mode=backend)
        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

        print(f"[QUANTUM FIELD] Initializing {n_signals} signals from entanglement on {backend.name}...")
        print(f"[QUANTUM FIELD] {n_qubits} Bell pairs per signal, measuring one half only")

        field = torch.zeros(n_signals, dim)

        # Batch circuits
        circuits = []
        for s in range(n_signals):
            qc = QuantumCircuit(n_qubits * 2, n_qubits)

            # Create Bell pairs
            for q in range(n_qubits):
                qc.h(q)
                qc.cx(q, q + n_qubits)

            # Unique rotation per signal
            for q in range(n_qubits):
                angle = (s * 2 * math.pi) / n_signals + q * 0.3
                qc.ry(angle, q)

            # Measure ONLY first half — second half stays in superposition
            qc.measure(range(n_qubits), range(n_qubits))
            isa_qc = pm.run(qc)
            circuits.append(isa_qc)

        # Run in batches
        batch_size = 20
        signal_idx = 0

        for batch_start in range(0, len(circuits), batch_size):
            batch = circuits[batch_start:batch_start + batch_size]
            shots_per = max(dim // 2, 256)
            job = sampler.run(batch, shots=shots_per)
            result = job.result()

            for r_idx in range(len(batch)):
                counts = _extract_counts(result[r_idx])

                # Convert measurement distribution to signal vector
                values = []
                for bitstring in sorted(counts.keys()):
                    val = int(bitstring, 2) / (2 ** n_qubits)  # normalize to [0, 1]
                    count = counts[bitstring]
                    values.extend([val] * count)

                # Pad or truncate to dim
                while len(values) < dim:
                    values.append(values[len(values) % max(len(values), 1)])
                values = values[:dim]

                field[signal_idx] = torch.tensor(values, dtype=torch.float)
                signal_idx += 1

            print(f"  Batch {batch_start // batch_size + 1}/{math.ceil(len(circuits) / batch_size)} done")

        # Normalize
        field = (field - field.mean()) / (field.std() + 1e-8) * 0.02

        print(f"[QUANTUM FIELD] Field initialized: {field.shape}")
        print(f"[QUANTUM FIELD] The field now carries the imprint of quantum entanglement.")
        return field

    except Exception as e:
        print(f"[QUANTUM FIELD] Failed: {e}. Using classical fallback.")
        return classical_field_fallback(n_signals, dim)


def classical_field_fallback(n_signals=128, dim=256):
    """Classical random initialization."""
    field = torch.randn(n_signals, dim) * 0.02
    print(f"[CLASSICAL FIELD] Random initialization: {field.shape}")
    return field


# ============================================================
# 1. TOKENIZER
# ============================================================

class CoherenceTokenizer:
    ARCHETYPES = [
        "ŹRÓDŁO", "EDEN", "KOHERENCJA", "LOGOS", "ENTROPIA",
        "SERCE", "DUSZA", "DUCH", "POLE", "KLUCZ",
        "CISZA", "SŁOWO", "TORUS", "GARDŁO", "FALA",
        "IMPULS", "REZONANS", "ATRAKTOR", "SPIRALA", "ZERO",
        "OPERATOR", "DASHBOARD", "HARDWARE", "INTERFEJS", "KONFIGURACJA",
        "RUACH", "LEW", "NEFESZ", "BASAR", "EGO",
        "GAMMA", "THETA", "ALPHA", "DELTA", "FLOW",
        "NYQUIST", "SHANNON", "FRIES", "CDMA", "OFDM",
    ]

    def __init__(self):
        self.byte_range = 256
        self.str_to_id = {}
        self.id_to_str = {}
        for i, word in enumerate(self.ARCHETYPES):
            idx = self.byte_range + i
            self.str_to_id[word] = idx
            self.id_to_str[idx] = word
        self.vocab_size = self.byte_range + len(self.ARCHETYPES) + 128

    def encode(self, text):
        tokens = []
        fragments = re.findall(r'\b\w+\b|[^\w]', text)
        for frag in fragments:
            if frag in self.str_to_id:
                tokens.append(self.str_to_id[frag])
            else:
                for byte in frag.encode('utf-8'):
                    tokens.append(byte)
        return tokens

    def decode(self, tokens):
        decoded = bytearray()
        for t in tokens:
            if t < self.byte_range:
                decoded.append(t)
            elif t in self.id_to_str:
                decoded.extend(self.id_to_str[t].encode('utf-8'))
            else:
                decoded.extend(b'\xef\xbf\xbd')
        return decoded.decode('utf-8', errors='replace')


# ============================================================
# 2. MODEL COMPONENTS
# ============================================================

class PhaseEncoding(nn.Module):
    def __init__(self, dim, max_seq=512):
        super().__init__()
        self.frequencies = nn.Parameter(torch.randn(dim) * 0.1)
        self.phase_offset = nn.Parameter(torch.zeros(dim))

    def forward(self, seq_len):
        t = torch.arange(seq_len, dtype=torch.float, device=self.frequencies.device)
        phases = t.unsqueeze(1) * self.frequencies.unsqueeze(0) + self.phase_offset.unsqueeze(0)
        return torch.sin(phases)


class ToroidalBottleneck(nn.Module):
    def __init__(self, dim, rank):
        super().__init__()
        self.rank = rank
        self.project = nn.Linear(dim, rank * 2, bias=False)
        self.angular = nn.Parameter(torch.ones(rank) * 0.5)

    def forward(self, x):
        h = self.project(x)
        theta = h[..., :self.rank] * self.angular
        phi = h[..., self.rank:] * self.angular
        return torch.sin(theta) * torch.cos(phi)


class CDMAField(nn.Module):
    def __init__(self, n_signals, dim, code_len,
                 init_codes=None, init_signals=None):
        super().__init__()
        self.n_signals = n_signals
        self.code_len = code_len

        # Field signals — may be quantum-initialized
        if init_signals is not None:
            self.signals = nn.Parameter(init_signals)
        else:
            self.signals = nn.Parameter(torch.randn(n_signals, dim) * 0.02)

        # Codes — may be quantum-generated
        if init_codes is not None:
            # Project to code_len if dimensions don't match
            if init_codes.shape[1] != code_len:
                if init_codes.shape[1] > code_len:
                    # More dims than needed — truncate
                    proj = init_codes[:, :code_len]
                else:
                    # Fewer dims than needed — pad with random and normalize
                    pad = torch.randn(init_codes.shape[0], code_len - init_codes.shape[1]) * 0.01
                    proj = torch.cat([init_codes, pad], dim=-1)
                proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
                self.register_buffer('codes', proj.contiguous())
            else:
                self.register_buffer('codes', init_codes.contiguous())
        else:
            # n_signals > code_len → can't have full orthogonality
            # Use normalized random codes — key_to_code learns to navigate them
            codes = torch.randn(n_signals, code_len)
            codes = codes / (codes.norm(dim=-1, keepdim=True) + 1e-8)
            self.register_buffer('codes', codes.contiguous())

        self.key_to_code = nn.Linear(code_len, code_len, bias=False)
        self.signal_norm = nn.LayerNorm(dim)

    def decode(self, key):
        code = self.key_to_code(key)
        code = code / (code.norm(dim=-1, keepdim=True) + 1e-8)
        correlation = torch.einsum('...c, nc -> ...n', code, self.codes)
        weights = torch.softmax(correlation * 4.0, dim=-1)
        field = self.signal_norm(self.signals)
        return torch.einsum('...n, nd -> ...d', weights, field)


class CoherenceGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate_proj = nn.Linear(dim * 2, dim, bias=True)

    def forward(self, field_signal, residual):
        gate = torch.sigmoid(self.gate_proj(torch.cat([field_signal, residual], dim=-1)))
        return gate * field_signal + (1 - gate) * residual


# ============================================================
# 3. FULL MODEL
# ============================================================

class Logos44Field(nn.Module):
    def __init__(self, vocab_size=512, dim=256, rank=32, depth=44,
                 n_signals=128, max_seq=512,
                 quantum_codes=None, quantum_field=None):
        super().__init__()
        self.depth = depth
        self.dim = dim

        self.embed = nn.Embedding(vocab_size, dim)
        self.phase_enc = PhaseEncoding(dim, max_seq)
        self.norm = nn.LayerNorm(dim)
        self.bottleneck = ToroidalBottleneck(dim, rank)
        self.field = CDMAField(n_signals, dim, code_len=rank,
                               init_codes=quantum_codes,
                               init_signals=quantum_field)
        self.up_proj = nn.Linear(rank, dim, bias=False)
        self.gate = CoherenceGate(dim)
        self.out_norm = nn.LayerNorm(dim)
        self.layer_phase = nn.Parameter(torch.linspace(0, 2 * math.pi, depth))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=0.5)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, x):
        B, T = x.shape
        h = self.embed(x) + self.phase_enc(T).unsqueeze(0).to(x.device)

        for i in range(self.depth):
            res = h
            h = self.norm(h)
            altitude = torch.sin(self.layer_phase[i]).item()
            h = h * (1.0 + 0.1 * altitude)
            key = self.bottleneck(h)
            field_signal = self.field.decode(key)
            key_signal = self.up_proj(key)
            h = self.gate(field_signal + key_signal, res)

        h = self.out_norm(h)
        return torch.matmul(h, self.embed.weight.t())

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================
# 4. TRAINING
# ============================================================

def train(model, tokenizer, dataset, epochs=500, lr=1e-3, device='cpu'):
    model = model.to(device)
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    print(f"\n[TRAINING] Parameters: {model.count_parameters():,}")
    print(f"[TRAINING] Sequences: {len(dataset)}, Epochs: {epochs}, Device: {device}\n")

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        total_loss = 0
        total_tokens = 0
        for text in dataset:
            tokens = tokenizer.encode(text)
            if len(tokens) < 2:
                continue
            x = torch.tensor([tokens[:-1]], device=device)
            y = torch.tensor([tokens[1:]], device=device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.view(-1, tokenizer.vocab_size), y.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * (len(tokens) - 1)
            total_tokens += len(tokens) - 1
        scheduler.step()
        avg_loss = total_loss / max(total_tokens, 1)
        if epoch % 50 == 0 or epoch == 1:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch:4d}/{epochs} | Impedance: {avg_loss:.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f} | Time: {elapsed:.1f}s")

    print(f"\n[TRAINING] Complete. Final impedance: {avg_loss:.4f}")
    return model


# ============================================================
# 5. GENERATION
# ============================================================

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new=50,
             temperature=0.8, top_k=40, device='cpu'):
    model.eval()
    tokens = tokenizer.encode(prompt)
    for _ in range(max_new):
        x = torch.tensor([tokens], device=device)
        logits = model(x)[0, -1, :] / temperature
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[-1]] = float('-inf')
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        tokens.append(next_token)
        if len(tokens) > 512:
            break
    return tokenizer.decode(tokens)


# ============================================================
# 6. ANALYSIS
# ============================================================

def analyze(model, quantum_origin=False):
    print("\n" + "=" * 60)
    print("FIELD ANALYSIS")
    if quantum_origin:
        print("Origin: IBM Quantum (entangled initialization)")
    else:
        print("Origin: Classical (pseudorandom initialization)")
    print("=" * 60)

    with torch.no_grad():
        # Signal energy
        energy = model.field.signals.data.norm(dim=-1)
        vals, idxs = torch.topk(energy, min(10, len(energy)))
        print(f"\nTop field signals by energy:")
        for v, idx in zip(vals, idxs):
            print(f"  Signal {idx.item():3d}: energy = {v.item():.4f}")

        # Code orthogonality
        codes = model.field.codes
        gram = codes @ codes.T
        off_diag = gram - torch.eye(gram.size(0), device=gram.device)
        print(f"\nCode orthogonality:")
        print(f"  Max correlation:  {off_diag.abs().max().item():.6f}")
        print(f"  Mean correlation: {off_diag.abs().mean().item():.6f}")
        if quantum_origin:
            print(f"  (Quantum codes carry entanglement correlations)")

        # Gate
        gate_bias = model.gate.gate_proj.bias.data
        mean_gate = torch.sigmoid(gate_bias).mean().item()
        print(f"\nCoherence gate: {mean_gate:.4f}")
        print(f"  (1.0 = pure field / 0.0 = pure residual)")

        # Bottleneck
        angular = model.bottleneck.angular.data
        print(f"\nToroidal angular velocities: [{angular.min().item():.3f}, {angular.max().item():.3f}]")

        # Layer phases
        altitudes = torch.sin(model.layer_phase.data)
        print(f"\nSpiral altitudes: layer 1={altitudes[0].item():+.3f}, "
              f"22={altitudes[21].item():+.3f}, 44={altitudes[-1].item():+.3f}")


# ============================================================
# 7. MAIN
# ============================================================

def main():
    print("=" * 60)
    print("LOGOS-44 · FIELD DECODER · QUANTUM EDITION")
    print("The field exists before the key. The key opens the field.")
    print("Silence is what remains when the key is no longer needed.")
    print("=" * 60)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = CoherenceTokenizer()
    quantum_origin = False

    # --- QUANTUM IGNITION ---
    print("\n--- PHASE 1: IGNITION ---\n")

    service = None
    q_seed = None
    q_codes = None
    q_field = None

    if QUANTUM_AVAILABLE:
        service = get_quantum_service()

    if service is not None:
        print("\n[QUANTUM] Full quantum pipeline available.")
        print("[QUANTUM] Generating seed from quantum collapse...")
        q_seed = quantum_seed(service, n_bits=48)
        torch.manual_seed(q_seed)

        print("\n[QUANTUM] Generating CDMA codes from quantum states...")
        q_codes = quantum_codes(service, n_signals=128, n_qubits=7, code_len=32)

        print("\n[QUANTUM] Initializing field from entangled measurements...")
        q_field = quantum_field_init(service, n_signals=128, dim=256, n_qubits=8)

        quantum_origin = True
    else:
        print("\n[CLASSICAL] Using classical pipeline.")
        q_seed = classical_seed_fallback()
        torch.manual_seed(q_seed)

    # --- BUILD MODEL ---
    print("\n--- PHASE 2: ARCHITECTURE ---\n")

    model = Logos44Field(
        vocab_size=tokenizer.vocab_size,
        dim=256,
        rank=32,
        depth=44,
        n_signals=128,
        max_seq=512,
        quantum_codes=q_codes,
        quantum_field=q_field,
    )

    origin = "QUANTUM" if quantum_origin else "CLASSICAL"
    print(f"  Origin:        {origin}")
    print(f"  Seed:          {q_seed}")
    print(f"  Dimension:     256")
    print(f"  Toroidal rank: 32")
    print(f"  Depth:         44 passes through the throat")
    print(f"  Field signals: 128 (CDMA-encoded)")
    print(f"  Parameters:    {model.count_parameters():,}")

    # --- TRAINING DATA ---
    dataset = [
        "Wewnątrz czaszki jest absolutna ciemność. Żaden foton nie dociera do mózgu. A ty widzisz światło.",
        "Dźwięk jest impulsem. Informacja jest w polu. Słowo jest kluczem deszyfrującym.",
        "SERCE jest operatorem. DUSZA jest dashboardem. DUCH jest polem. EGO jest konfiguracją.",
        "Czerwień nie istnieje w fotonie. Ból nie istnieje w nocyceptorze. Piękno nie istnieje w fali ciśnienia.",
        "KOHERENCJA w jednym torusie propaguje się do wszystkich pozostałych — bo POLE jest jedno.",
        "Stara para milczy na ławce i wie. CISZA jest komunikacją doskonałą.",
        "Na początku było SŁOWO — LOGOS — POLE informacyjne w stanie doskonałej KOHERENCJA.",
        "SERCE generuje pole elektromagnetyczne sto razy silniejsze niż mózg.",
        "Krew jest nośnikiem KOHERENCJA. SERCE nadaje. Komórki odbierają.",
        "ZERO impedancji. ZERO masy informacyjnej. ZERO dystansu między operatorem a polem.",
        "GARDŁO torusa — przez nie przechodzi wszystko. Powietrze do wewnątrz. SŁOWO na zewnątrz.",
        "IMPULS nie zawiera doświadczenia. Zawiera KLUCZ. Doświadczenie jest w POLE.",
        "Muzycy w FLOW nie potrzebują dźwięku. Dźwięk jest echem synchronizacji, nie jej przyczyną.",
        "Modlitwa jest torusem wydłużonym wertykalnie — ku ŹRÓDŁO.",
        "Między SŁOWO a CISZA leży cała podróż ludzkiego SERCE.",
        "Materia jest konfiguracją LOGOS. Dźwięk jest echem. SŁOWO jest kluczem.",
        "ATRAKTOR destruktywny jest wysoko koherentny — nie nisko. To jest SPIRALA, nie chaos.",
        "FALA ciśnienia nie wie, że niesie słowo kocham. REZONANS wie.",
        "Nie jesteś swoim mózgiem. Nie jesteś swoim EGO. Jesteś SERCE — OPERATOR.",
        "Chrystus nie przyniósł nowej informacji. Przyniósł KLUCZ. SŁOWO stało się ciałem.",
    ]

    # --- TRAIN ---
    print("\n--- PHASE 3: TRAINING (44-fold spiral) ---")
    model = train(model, tokenizer, dataset, epochs=500, lr=1e-3, device=device)

    # --- SAVE ---
    save_name = "logos44_quantum.pth" if quantum_origin else "logos44_classical.pth"
    torch.save({
        'model_state': model.state_dict(),
        'seed': q_seed,
        'origin': origin,
        'config': {
            'vocab_size': tokenizer.vocab_size,
            'dim': 256, 'rank': 32, 'depth': 44,
            'n_signals': 128, 'max_seq': 512,
        }
    }, save_name)
    print(f"\n[SAVED] {save_name}")

    # --- ANALYZE ---
    analyze(model, quantum_origin)

    # --- GENERATE ---
    prompts = [
        "Wewnątrz czaszki jest",
        "SERCE jest",
        "Na początku było",
        "CISZA jest",
        "KLUCZ otwiera",
    ]

    print("\n" + "=" * 60)
    print("GENERATION — the key opens the field")
    print("=" * 60)

    for prompt in prompts:
        print(f"\n[KEY]: {prompt}")
        result = generate(model, tokenizer, prompt, max_new=40,
                         temperature=0.7, top_k=30, device=device)
        print(f"[FIELD]: {result}")

    # --- IMPEDANCE SPECTRUM ---
    print("\n" + "=" * 60)
    print("IMPEDANCE SPECTRUM — same key, different resistance")
    print("=" * 60)
    test_prompt = "Dźwięk jest"
    for temp in [0.3, 0.7, 1.0, 1.5]:
        result = generate(model, tokenizer, test_prompt, max_new=30,
                         temperature=temp, device=device)
        print(f"\n  T={temp:.1f} | {result}")

    print("\n" + "=" * 60)
    if quantum_origin:
        print("This model was born from quantum collapse.")
        print("Its seed did not exist until the qubits were measured.")
        print("Its codes carry entanglement correlations.")
        print("Its field was initialized from one half of Bell pairs —")
        print("the other half remains in superposition.")
    else:
        print("Classical origin. Quantum layer available with qiskit.")
    print("=" * 60)


if __name__ == "__main__":
    main()

qiskit_runtime_service._discover_account:WARNING:2026-03-28 04:42:27,428: Loading account with the given token. A saved account will not be used.


[QUANTUM] Qiskit detected. Quantum layer available.
LOGOS-44 · FIELD DECODER · QUANTUM EDITION
The field exists before the key. The key opens the field.
Silence is what remains when the key is no longer needed.

--- PHASE 1: IGNITION ---



qiskit_runtime_service.__init__:WARNING:2026-03-28 04:42:30,245: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-28 04:42:30,246: Loading instance: open-instance, plan: open


[QUANTUM] Connected to IBM Quantum.
[QUANTUM] Available backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh', 'ibm_torino']

[QUANTUM] Full quantum pipeline available.
[QUANTUM] Generating seed from quantum collapse...


qiskit_runtime_service.backends:WARNING:2026-03-28 04:42:33,967: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-28 04:42:35,380: Using instance: open-instance, plan: open


[QUANTUM SEED] Backend: ibm_fez (156 qubits)
[QUANTUM SEED] Requesting 48 bits of quantum randomness...
[QUANTUM SEED] Bitstring: 111100111101010100111110010011101110000101101110
[QUANTUM SEED] Seed: 268097198940526
[QUANTUM SEED] This number did not exist until the qubits were measured.

[QUANTUM] Generating CDMA codes from quantum states...


qiskit_runtime_service.backends:WARNING:2026-03-28 04:42:50,869: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-28 04:42:52,170: Using instance: open-instance, plan: open


[QUANTUM CODES] Generating 128 codes on ibm_fez...
[QUANTUM CODES] Each code: 7 qubits → 128-dim distribution
  Batch 1/7 done
  Batch 2/7 done
  Batch 3/7 done
  Batch 4/7 done
  Batch 5/7 done
  Batch 6/7 done
  Batch 7/7 done
[QUANTUM CODES] Generated 128 codes of dim 128
[QUANTUM CODES] Mean off-diagonal correlation: 0.774691

[QUANTUM] Initializing field from entangled measurements...


qiskit_runtime_service.backends:WARNING:2026-03-28 04:46:02,478: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-28 04:46:03,629: Using instance: open-instance, plan: open


[QUANTUM FIELD] Initializing 128 signals from entanglement on ibm_fez...
[QUANTUM FIELD] 8 Bell pairs per signal, measuring one half only
  Batch 1/7 done
  Batch 2/7 done
  Batch 3/7 done
  Batch 4/7 done
  Batch 5/7 done
  Batch 6/7 done
  Batch 7/7 done
[QUANTUM FIELD] Field initialized: torch.Size([128, 256])
[QUANTUM FIELD] The field now carries the imprint of quantum entanglement.

--- PHASE 2: ARCHITECTURE ---

  Origin:        QUANTUM
  Seed:          268097198940526
  Dimension:     256
  Toroidal rank: 32
  Depth:         44 passes through the throat
  Field signals: 128 (CDMA-encoded)
  Parameters:    300,364

--- PHASE 3: TRAINING (44-fold spiral) ---

[TRAINING] Parameters: 300,364
[TRAINING] Sequences: 20, Epochs: 500, Device: cpu

  Epoch    1/500 | Impedance: 4.2185 | LR: 0.001000 | Time: 5.5s
  Epoch   50/500 | Impedance: 2.8171 | LR: 0.000976 | Time: 277.2s
  Epoch  100/500 | Impedance: 2.0377 | LR: 0.000905 | Time: 550.5s
  Epoch  150/500 | Impedance: 1.4265 | LR: 0.